In [1]:
import pandas as pd
import numpy as np

file_path = "C:/Users/APURBA ROY/Desktop/SR_Revision/JUNE_Revision/Evaluation of Explainable AI (XAI) in Medical Imaging. (Responses).xlsx"

df = pd.read_excel(file_path)

print("Data shape:", df.shape)
df.head()

Data shape: (49, 25)


,Timestamp,What is your current year of study?,Have you used AI-based tools in diagnosis or training before?,Image 1: Which map best describes the highlighted abnormality or clinical feature?,Grad-CAM Explanation—Case 1\nHow well does Grad-CAM highlight the relevant diagnostic region?,LIME Explanation—Case 1\nHow well does LIME highlight the relevant diagnostic region?,SHAP Explanation—Case 1\nHow well does SHAP highlight the relevant diagnostic region?,"Image 2: From the maps below, which one best isolates the relevant clinical region?",Grad-CAM Explanation—Case 2\nHow well does Grad-CAM highlight the relevant diagnostic region?,LIME Explanation—Case 2\nHow well does LIME highlight the relevant diagnostic region?,...,Which model’s map do you trust more?,Rate the reliability of ResNet50.,Rate the reliability of VGG16.,Rate the reliability of InceptionV3.,Rate the reliability of Pathologist.,The explanation correctly highlighted the diagnostic region.,The explanation made the AI’s decision easier to understand.,The visualization increased my trust in the AI system.,I would find this explanation helpful in a real diagnostic setting.,Your Name:
0,2025-05-10 22:40:49.835,Non-Medical Student,Yes,LIME,3,4,1,LIME,3,4,...,"ResNet50, Manually Marked by Pathologist",4,3,3,5,3,4,3,4,Gobinda Chandra Sarker
1,2025-05-10 22:46:00.069,Non-Medical Student,Yes,GRAD_CAM,4,3,2,GRAD_CAM,3,3,...,"ResNet50, VGG16, InceptionV3",3,3,3,4,3,3,3,3,Protik Barua
2,2025-05-10 22:54:37.275,Non-Medical Student,No,GRAD_CAM,4,3,4,GRAD_CAM,4,3,...,VGG16,4,4,3,4,3,3,3,3,Imtiaz Ahmed
3,2025-05-10 22:55:48.521,Non-Medical Student,Yes,GRAD_CAM,5,3,4,LIME,3,5,...,"ResNet50, VGG16",5,4,3,2,3,4,3,4,Md. Mahbub Ul Haque
4,2025-05-10 23:12:05.104,Non-Medical Student,No,GRAD_CAM,4,2,2,GRAD_CAM,4,3,...,"ResNet50, VGG16, InceptionV3, Manually Marked ...",3,3,3,3,3,3,3,3,Sadia Shara Toma


In [2]:
group_col = "What is your current year of study? "

df["Participant_Group"] = np.where(
    df[group_col].astype(str).str.contains("Non-Medical", case=False, na=False),
    "Non-medical",
    "Medical"
)

print(df["Participant_Group"].value_counts())

Participant_Group
Medical        28
Non-medical    21
Name: count, dtype: int64


In [3]:
rating_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Number of rating items:", len(rating_cols))
for col in rating_cols:
    print(col)

Number of rating items: 17
Grad-CAM Explanation—Case 1
How well does Grad-CAM highlight the relevant diagnostic region?
LIME Explanation—Case 1
How well does LIME highlight the relevant diagnostic region?
SHAP Explanation—Case 1
How well does SHAP highlight the relevant diagnostic region?
Grad-CAM Explanation—Case 2
How well does Grad-CAM highlight the relevant diagnostic region?
LIME Explanation—Case 2
How well does LIME highlight the relevant diagnostic region?
SHAP Explanation—Case 2
How well does SHAP highlight the relevant diagnostic region?
Grad-CAM Explanation—Case 3
How well does Grad-CAM highlight the relevant diagnostic region?
LIME Explanation—Case 3

How well does LIME highlight the relevant diagnostic region?
SHAP Explanation—Case 3
How well does SHAP highlight the relevant diagnostic region?
Rate the reliability of ResNet50.
Rate the reliability of VGG16.
Rate the reliability of InceptionV3.
Rate the reliability of Pathologist.
The explanation correctly highlighted the di

In [4]:
group_means = df.groupby("Participant_Group")[rating_cols].mean().T

group_means = group_means[["Medical", "Non-medical"]]

group_means.head()

Participant_Group,Medical,Non-medical
Grad-CAM Explanation—Case 1\nHow well does Grad-CAM highlight the relevant diagnostic region?,3.785714,4.285714
LIME Explanation—Case 1\nHow well does LIME highlight the relevant diagnostic region?,3.607143,3.142857
SHAP Explanation—Case 1\nHow well does SHAP highlight the relevant diagnostic region?,3.035714,3.047619
Grad-CAM Explanation—Case 2\nHow well does Grad-CAM highlight the relevant diagnostic region?,3.500000,3.714286
LIME Explanation—Case 2\nHow well does LIME highlight the relevant diagnostic region?,3.785714,3.142857


In [5]:
def calculate_icc_2(data_matrix):
    """
    ICC(2,1) and ICC(2,k): two-way random-effects ICC.
    Rows = targets/items
    Columns = raters/groups
    """
    X = np.asarray(data_matrix, dtype=float)

    n, k = X.shape   # n = targets, k = raters/groups

    grand_mean = X.mean()
    row_means = X.mean(axis=1)
    col_means = X.mean(axis=0)

    SSR = k * np.sum((row_means - grand_mean) ** 2)
    SSC = n * np.sum((col_means - grand_mean) ** 2)
    SSE = np.sum((X - row_means[:, None] - col_means[None, :] + grand_mean) ** 2)

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    ICC_2_1 = (MSR - MSE) / (MSR + (k - 1) * MSE + (k * (MSC - MSE) / n))
    ICC_2_k = (MSR - MSE) / (MSR + ((MSC - MSE) / n))

    return {
        "Number of items": n,
        "Number of groups": k,
        "MSR": MSR,
        "MSC": MSC,
        "MSE": MSE,
        "ICC(2,1)": ICC_2_1,
        "ICC(2,k)": ICC_2_k
    }

icc_results = calculate_icc_2(group_means.values)

icc_results

{'Number of items': 17,
 'Number of groups': 2,
 'MSR': 0.1783400860344138,
 'MSC': 0.11905178738161847,
 'MSE': 0.06268132252901158,
 'ICC(2,1)': 0.46701900257519435,
 'ICC(2,k)': 0.6366911427260215}

In [6]:
icc_table = pd.DataFrame([icc_results])

icc_table[["ICC(2,1)", "ICC(2,k)"]] = icc_table[["ICC(2,1)", "ICC(2,k)"]].round(3)

icc_table

,Number of items,Number of groups,MSR,MSC,MSE,"ICC(2,1)","ICC(2,k)"
0,17,2,0.17834,0.119052,0.062681,0.467,0.637


In [7]:
group_means.to_csv("medical_vs_nonmedical_group_means.csv")
icc_table.to_csv("icc_medical_vs_nonmedical.csv", index=False)

print("Saved:")
print("medical_vs_nonmedical_group_means.csv")
print("icc_medical_vs_nonmedical.csv")

Saved:
medical_vs_nonmedical_group_means.csv
icc_medical_vs_nonmedical.csv
